In [6]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import geopandas as gpd
import re
import matplotlib.ticker as ticker
from matplotlib.patheffects import withStroke
import plotly.express as px

In [7]:
df = pd.read_csv('cleaned_new_data.csv')

In [9]:
df = df.set_index(["Year", "Province"]).drop(columns=["row_id"])

# --- General school info ---
df_general = df[[
    "num_schools", "num_disadvantaged_schools", "num_classes", "classes_in_pagoda",
    "enrollment_total", "enrollment_girl", "repeaters_total", "repeaters_girl",
    "teaching_staff_total", "teaching_staff_female",
    "non_teaching_staff_total", "non_teaching_staff_female",
    "total_staff_total", "total_staff_female",
]]

# --- Schools by level ---
df_schools_by_level = df[[
    "num_schools_preschool", "num_schools_primary", "num_schools_college", "num_schools_lycee",
    "two_shift_schools_lycee", "floating_schools_lycee", "schools_in_pagoda_lycee", "attached_preschool_lycee",
    "schools_without_water", "schools_without_latrine",
    "principal_avg_age", "principal_avg_service_years", "principal_upper_sec_plus_edu", "principal_female",
]]

# --- Cluster & satellite schools ---
df_cluster = df[[
    "num_primary_schools", "num_cluster_schools", "satellite_schools", "satellite_classes",
    "pagoda_preschool_schools", "pagoda_preschool_classes",
    "pagoda_primary_schools", "pagoda_primary_classes",
    "pagoda_junior_high_schools", "pagoda_junior_high_classes",
    "pagoda_senior_high_schools", "pagoda_senior_high_classes",
]]

# --- Grade 1-3 enrollment & repeaters ---
df_g1_g3 = df[[
    "g1_enrollment_total", "g1_enrollment_girl", "g1_repeaters_total", "g1_repeaters_girl",
    "g2_enrollment_total", "g2_enrollment_girl", "g2_repeaters_total", "g2_repeaters_girl",
    "g3_enrollment_total", "g3_enrollment_girl", "g3_repeaters_total", "g3_repeaters_girl",
]]

# --- Grade 4-6 enrollment & repeaters ---
df_g4_g6 = df[[
    "g4_enrollment_total", "g4_enrollment_girl", "g4_repeaters_total", "g4_repeaters_girl",
    "g5_enrollment_total", "g5_enrollment_girl", "g5_repeaters_total", "g5_repeaters_girl",
    "g6_enrollment_total", "g6_enrollment_girl", "g6_repeaters_total", "g6_repeaters_girl",
]]

# --- Grade 7-9 enrollment & repeaters ---
df_g7_g9 = df[[
    "g7_enrollment_total", "g7_enrollment_girl", "g7_repeaters_total", "g7_repeaters_girl",
    "g8_enrollment_total", "g8_enrollment_girl", "g8_repeaters_total", "g8_repeaters_girl",
    "g9_enrollment_total", "g9_enrollment_girl", "g9_repeaters_total", "g9_repeaters_girl",
]]

# --- Grade 10-12 enrollment & repeaters ---
df_g10_g12 = df[[
    "g10_enrollment_total", "g10_enrollment_girl", "g10_repeaters_total", "g10_repeaters_girl",
    "g11_enrollment_total", "g11_enrollment_girl", "g11_repeaters_total", "g11_repeaters_girl",
    "g12_enrollment_total", "g12_enrollment_girl", "g12_repeaters_total", "g12_repeaters_girl",
]]

# --- Intake & over-age enrollment ---
df_intake = df[[
    "g1_intake_total", "g1_intake_aged6",
    "primary_enrollment_total", "primary_enrollment_aged11plus",
    "g7_intake_total", "g7_intake_aged12",
    "lower_sec_enrollment_total", "lower_sec_enrollment_aged14plus",
    "g10_intake_total", "g10_intake_aged15",
    "upper_sec_enrollment_total", "upper_sec_enrollment_aged17plus",
    "pct_overage_enrollment_primary", "pct_overage_enrollment_lower_sec", "pct_overage_enrollment_upper_sec",
]]

# --- Staff education level ---
df_staff_edu = df[[
    "teaching_staff_edu_primary", "teaching_staff_edu_lower_sec", "teaching_staff_edu_upper_sec",
    "teaching_staff_edu_graduate", "teaching_staff_edu_postgrad", "teaching_staff_edu_phd",
    "non_teaching_staff_edu_primary", "non_teaching_staff_edu_lower_sec", "non_teaching_staff_edu_upper_sec",
    "non_teaching_staff_edu_graduate", "non_teaching_staff_edu_postgrad", "non_teaching_staff_edu_phd",
    "teaching_staff_no_pedagogy_primary", "teaching_staff_no_pedagogy_lower_sec", "teaching_staff_no_pedagogy_upper_sec",
    "teaching_staff_no_pedagogy_graduate", "teaching_staff_no_pedagogy_postgrad", "teaching_staff_no_pedagogy_phd",
]]

# --- Buildings & infrastructure ---
df_buildings = df[[
    "num_buildings_total", "num_rooms_total",
    "concrete_brick_buildings", "concrete_brick_rooms",
    "wooden_buildings", "wooden_rooms",
    "bamboo_buildings", "bamboo_rooms",
    "buildings_repaired", "buildings_constructed",
    "buildings_poor_floor", "buildings_poor_roof", "buildings_poor_wall",
]]

# --- Classrooms ---
df_classrooms = df[[
    "num_classrooms", "classrooms_in_pagoda", "classrooms_repaired", "classrooms_constructed",
    "classrooms_poor_floor", "classrooms_poor_roof", "classrooms_poor_wall",
    "schools_with_office", "schools_with_library",
]]

# --- School area & sports ---
df_area_sports = df[[
    "school_area_land_m2", "school_area_playground_m2", "classroom_area_per_student", "preschool_with_sport_facility",
    "sport_volleyball", "sport_football", "sport_basketball", "sport_climbing_ropes",
    "sport_shotput", "sport_high_jump", "sport_long_jump", "sport_running",
]]

# --- Community & funding ---
df_community = df[[
    "parent_assoc_exists", "parent_assoc_held_meeting", "parent_assoc_members_total", "parent_assoc_members_female",
    "community_teachers", "teaching_monks",
    "pb_fund_per_student_riel", "pb_fund_per_school_riel",
    "funding_school_income", "funding_community", "funding_govt_building", "funding_abroad", "funding_ios_ngos",
]]

# --- Population by age group ---
df_population = df[[
    "pop_aged6_total", "pop_aged6_girl",
    "pop_aged6_11_total", "pop_aged6_11_girl",
    "pop_aged12_14_total", "pop_aged12_14_girl",
    "pop_aged15_17_total", "pop_aged15_17_girl",
    "sex_ratio_aged6", "sex_ratio_aged6_11", "sex_ratio_aged12_14", "sex_ratio_aged15_17",
]]

# --- School efficiency ratios ---
df_efficiency = df[[
    "pupils_per_school", "teachers_per_school", "staff_per_school",
    "buildings_per_school", "rooms_per_school", "classrooms_per_school", "classes_per_school",
    "pct_schools_two_shift", "pct_schools_in_pagoda", "pct_schools_without_water", "pct_schools_without_toilet",
]]

# --- Pupil ratios ---
df_pupil_ratios = df[[
    "pupil_teacher_ratio", "pupil_staff_ratio", "pupil_class_ratio", "pupil_classroom_ratio",
    "classes_per_classroom", "classroom_area_per_pupil_m2",
    "pct_students_preschool", "pct_students_primary", "pct_students_lower_sec", "pct_students_upper_sec",
    "pct_non_teaching_staff", "pct_female_staff",
]]

# --- Repeater & over-age rates ---
df_repeater_rates = df[[
    "pct_repeaters_total_primary", "pct_repeaters_total_lower_sec", "pct_repeaters_total_upper_sec",
    "pct_overage_total_primary", "pct_overage_total_lower_sec", "pct_overage_total_upper_sec",
    "pct_repeaters_girl_primary", "pct_repeaters_girl_lower_sec", "pct_repeaters_girl_upper_sec",
    "pct_overage_girl_primary", "pct_overage_girl_lower_sec", "pct_overage_girl_upper_sec",
]]

# --- Admission & transition rates ---
df_admission = df[[
    "gross_admission_rate_total", "gross_admission_rate_girl",
    "net_admission_rate_total", "net_admission_rate_girl",
    "pct_overage_admission_total", "pct_overage_admission_girl",
    "transition_rate_lower_sec_total", "transition_rate_lower_sec_girl",
    "transition_rate_upper_sec_total", "transition_rate_upper_sec_girl",
]]

# --- Grade promotion / repetition / dropout (G1-G4) ---
df_flow_g1_g4 = df[[
    "g1_promotion", "g1_repetition", "g1_dropout",
    "g2_promotion", "g2_repetition", "g2_dropout",
    "g3_promotion", "g3_repetition", "g3_dropout",
    "g4_promotion", "g4_repetition", "g4_dropout",
]]

# --- Grade promotion / repetition / dropout (G5-G8) ---
df_flow_g5_g8 = df[[
    "g5_promotion", "g5_repetition", "g5_dropout",
    "g6_promotion", "g6_repetition", "g6_dropout",
    "g7_promotion", "g7_repetition", "g7_dropout",
    "g8_promotion", "g8_repetition", "g8_dropout",
]]

# --- Grade promotion / repetition / dropout (G9-G12) ---
df_flow_g9_g12 = df[[
    "g9_promotion", "g9_repetition", "g9_dropout",
    "g10_promotion", "g10_repetition", "g10_dropout",
    "g11_promotion", "g11_repetition", "g11_dropout",
    "g12_promotion", "g12_repetition", "g12_dropout",
]]